In [14]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, fbeta_score,
    average_precision_score
)
from xgboost import XGBClassifier

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA = ROOT / "data"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

transactions = pd.read_csv(DATA / "transactions.csv")
customers = pd.read_csv(DATA / "customers.csv")
answers = pd.read_csv(DATA / "answers.csv")

print("Data loaded successfully.")

Data loaded successfully.


In [15]:
df = transactions.merge(customers, on="CustomerID")
df = df.merge(answers[["TransactionID", "IsFraud", "FraudCaseID"]],
              on="TransactionID")

df["TransactionDate"] = pd.to_datetime(df["TransactionDate"])
df["Hour"] = df["TransactionDate"].dt.hour
df["DayOfWeek"] = df["TransactionDate"].dt.dayofweek
df["IsNight"] = ((df["Hour"] < 6) | (df["Hour"] >= 22)).astype(int)

df["AmountToIncomeRatio"] = (
    df["AmountINR"] / df["MonthlyIncomeINR"]
)
df["IsDifferentCity"] = (
    df["City"] != df["HomeCity"]
).astype(int)

print("Features prepared:", len(df))

Features prepared: 10000


In [16]:
features = [
    "AmountINR", "AmountToIncomeRatio", "Hour", "DayOfWeek",
    "IsNight", "Age", "MonthlyIncomeINR",
    "FailedLoginAttempts", "FailedPaymentAttempts",
    "IsNewDevice", "IsNewBeneficiary", "IsDifferentCity",
    "PaymentMethod", "TransactionType", "Channel",
    "MerchantCategory", "City", "Occupation",
    "PreferredPaymentMethod"
]

# Give each fraud case one group; legitimate transactions stay separate
df["GroupID"] = df["FraudCaseID"].fillna("").apply(
    lambda x: "CASE_" + x if x else None
)
df["GroupID"] = df["GroupID"].fillna(
    "TXN_" + df["TransactionID"]
)

group_dates = (
    df.groupby("GroupID")["TransactionDate"]
    .max().sort_values()
)

groups = group_dates.index.tolist()
train_end = int(len(groups) * 0.70)
valid_end = int(len(groups) * 0.85)

train_groups = set(groups[:train_end])
valid_groups = set(groups[train_end:valid_end])
test_groups = set(groups[valid_end:])

train = df[df["GroupID"].isin(train_groups)]
valid = df[df["GroupID"].isin(valid_groups)]
test = df[df["GroupID"].isin(test_groups)]

X_train, y_train = train[features], train["IsFraud"]
X_valid, y_valid = valid[features], valid["IsFraud"]
X_test, y_test = test[features], test["IsFraud"]

print("Training:", len(train), "| Fraud:", y_train.sum())
print("Validation:", len(valid), "| Fraud:", y_valid.sum())
print("Testing:", len(test), "| Fraud:", y_test.sum())

Training: 6992 | Fraud: 60
Validation: 1504 | Fraud: 20
Testing: 1504 | Fraud: 20


In [17]:
numeric = [
    "AmountINR", "AmountToIncomeRatio", "Hour", "DayOfWeek",
    "IsNight", "Age", "MonthlyIncomeINR",
    "FailedLoginAttempts", "FailedPaymentAttempts",
    "IsNewDevice", "IsNewBeneficiary", "IsDifferentCity"
]

categorical = [c for c in features if c not in numeric]

def create_pipeline(model):
    preprocessing = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical)
    ])

    return Pipeline([
        ("preprocessing", preprocessing),
        ("model", model)
    ])

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        class_weight="balanced",
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=(y_train == 0).sum() / max(1, y_train.sum()),
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

trained_models = {}

for name, model in models.items():
    pipeline = create_pipeline(model)
    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline
    print(name, "trained successfully.")

Random Forest trained successfully.
XGBoost trained successfully.


In [18]:
results = []

for name, model in trained_models.items():
    probabilities = model.predict_proba(X_valid)[:, 1]

    thresholds = np.arange(0.05, 0.96, 0.01)

    best_threshold = max(
        thresholds,
        key=lambda t: fbeta_score(
            y_valid, probabilities >= t,
            beta=2, zero_division=0
        )
    )

    predictions = probabilities >= best_threshold

    results.append({
        "Model": name,
        "Threshold": best_threshold,
        "Precision": precision_score(
            y_valid, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_valid, predictions, zero_division=0
        ),
        "F2": fbeta_score(
            y_valid, predictions, beta=2, zero_division=0
        ),
        "PR-AUC": average_precision_score(y_valid, probabilities)
    })

comparison = pd.DataFrame(results).sort_values(
    ["F2", "PR-AUC"], ascending=False
)

display(comparison)

,Model,Threshold,Precision,Recall,F2,PR-AUC
1,XGBoost,0.12,0.842105,0.8,0.808081,0.820615
0,Random Forest,0.06,0.571429,0.8,0.740741,0.759546


In [19]:
best = comparison.iloc[0]
best_name = best["Model"]

joblib.dump({
    "model": trained_models[best_name],
    "features": features,
    "threshold": float(best["Threshold"]),
    "model_name": best_name
}, MODEL_DIR / "fraud_model.joblib")

# Save untouched test data for the next evaluation notebook
joblib.dump(
    {"X_test": X_test, "y_test": y_test},
    MODEL_DIR / "test_data.joblib"
)

print("Selected model:", best_name)
print("Saved model and threshold successfully.")

Selected model: XGBoost
Saved model and threshold successfully.
